# Importing the packages and data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

In [3]:
import sys
sys.path.insert(1, '../sar_dirichlet')
import dirichlet_regression

In [4]:
from func_test import cos_similarity, create_features_matrices, rmse_aitchison

# With two features

In [5]:
n_features = 2
n_classes = 3

In [6]:
np.random.seed(21)

beta = np.array([[0.  , 0. , .1],
                 [0.  , 1., -2.],
                 [0.  , -1., -2. ]])

gamma_var = np.array([2.,3.])

In [7]:
n_repeat = 100
list_n_samples = [50,200,1000]

In [8]:
cov_matrix = np.array([[1., 0.2], [0.2, 1.]])

# Estimation of the parameters

In [9]:
n_samples=200
rho=0.5

X,Z,W = create_features_matrices(n_samples,n_features,choice_W='random_distance',nneighbors=10,cov_mat=cov_matrix)
Z[:,0] = 1
M = np.identity(n_samples) - rho*W

mu = dirichlet_regression.compute_mu_spatial(X, beta, M)
#phi = np.exp(np.matmul(Z,gamma_var))
phi = 15*np.ones(n_samples)
alpha = mu*phi[:,None]

Y = np.array([np.random.dirichlet(alpha_i) for alpha_i in alpha])
Y = (Y*(n_samples-1)+1/n_classes)/n_samples

In [10]:
np.sum(Y,axis=0)

array([57.85925068, 57.33059709, 84.81015224])

In [11]:
%%time
reg_spatial = dirichlet_regression.dirichletRegressor(spatial=True, maxfun=5000)
reg_spatial.fit(X, Y, parametrization='alternative', Z=Z, W=W, fit_intercept=False, verbose=1)


CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Wall time: 899 ms


In [12]:
dirichlet_regression.dirichlet_loglikelihood(reg_spatial.mu,reg_spatial.phi,Y)

704.287556163925

In [13]:
reg_spatial.rho

0.5070775210741895

In [14]:
reg_spatial.inference(X, Y, Z, W, display=True)

-----
Estimated parameter beta_11 = 0.0296, se = 0.027, CI 95% = [-0.0233 ; 0.0825],  p-value = 0.2731
-----
Estimated parameter beta_12 = 0.1591, se = 0.0263, CI 95% = [0.1075 ; 0.2106],  p-value = 0.0
-----
Estimated parameter beta_21 = 0.9411, se = 0.0549, CI 95% = [0.8335 ; 1.0487],  p-value = 0.0
-----
Estimated parameter beta_22 = -1.5726, se = 0.0699, CI 95% = [-1.7095 ; -1.4356],  p-value = 0.0
-----
Estimated parameter beta_31 = -1.0498, se = 0.0539, CI 95% = [-1.1555 ; -0.944],  p-value = 0.0
-----
Estimated parameter beta_32 = -1.8677, se = 0.0616, CI 95% = [-1.9884 ; -1.7469],  p-value = 0.0
-----
Estimated parameter gamma_1 = 2.9348, se = 0.1376, CI 95% = [2.6651 ; 3.2045],  p-value = 0.0
-----
Estimated parameter gamma_2 = -0.0118, se = 0.2176, CI 95% = [-0.4383 ; 0.4147],  p-value = 0.9567
-----
Estimated parameter rho = 0.5071, se = 0.0208, CI 95% = [0.4664 ; 0.5478],  p-value = 0.0


In [15]:
np.sqrt(sys.float_info.epsilon)

1.4901161193847656e-08

In [16]:
sys.float_info.epsilon

2.220446049250313e-16

In [17]:
#h = np.sqrt(sys.float_info.epsilon)
h = 10e-6
n = len(X)

In [18]:
M = np.identity(n) - (reg_spatial.rho)*W

In [19]:
M_h_moins = np.identity(n) - (reg_spatial.rho-h)*W
M_h_plus = np.identity(n) - (reg_spatial.rho+h)*W

In [20]:
zeros_col = np.zeros((reg_spatial.beta.shape[0], 1))
beta_hat = np.hstack([zeros_col, reg_spatial.beta])

In [21]:
mu_h_moins = dirichlet_regression.compute_mu_spatial(X, beta_hat, M_h_moins)
mu_h_plus = dirichlet_regression.compute_mu_spatial(X, beta_hat, M_h_plus)
mu_hat = dirichlet_regression.compute_mu_spatial(X, beta_hat, M)

In [22]:
ll_h_moins = dirichlet_regression.dirichlet_loglikelihood(mu_h_moins,reg_spatial.phi,Y)
ll_h_plus = dirichlet_regression.dirichlet_loglikelihood(mu_h_plus,reg_spatial.phi,Y)
ll_hat = dirichlet_regression.dirichlet_loglikelihood(mu_hat,reg_spatial.phi,Y)

In [23]:
ll_h_moins

704.2875560279472

In [24]:
ll_h_plus

704.2875559919063

In [25]:
dirichlet_regression.derivative_wrt_rho(reg_spatial.mu, reg_spatial.phi, beta_hat, M, W, X, Y)

-0.0018016495194785875

In [26]:
ll = dirichlet_regression.dirichlet_loglikelihood(reg_spatial.mu,reg_spatial.phi,Y)

In [27]:
(ll_h_plus - 2*ll_hat + ll_h_moins)/(h**2)

-3079.966290897573

In [28]:
(ll_h_plus - ll_hat)/h

-0.017201875834871316

In [29]:
reg_spatial.compute_hessian(X,Y,Z,W)

In [30]:
np.diag(reg_spatial.hess)

array([-2646.46633003, -2550.83614106,  -524.89835983,  -364.06579574,
        -661.5944185 ,  -615.30963476,  -254.59498532,   -83.58066802,
       -3079.96842806])

In [31]:
np.diag(reg_spatial.hess)

array([-2646.46633003, -2550.83614106,  -524.89835983,  -364.06579574,
        -661.5944185 ,  -615.30963476,  -254.59498532,   -83.58066802,
       -3079.96842806])

In [32]:
#h = np.sqrt(sys.float_info.epsilon)
h = 10e-7

In [33]:
zeros_col = np.zeros((reg_spatial.beta.shape[0], 1))
beta_hat = np.hstack([zeros_col, reg_spatial.beta])

In [34]:
beta_h_plus = np.copy(beta_hat)
beta_h_plus[0,0] += h
beta_h_moins = np.copy(beta_hat)
beta_h_moins[0,0] -= h

In [35]:
mu_h_moins = dirichlet_regression.compute_mu_spatial(X, beta_h_moins, M)
mu_h_plus = dirichlet_regression.compute_mu_spatial(X, beta_h_plus, M)
mu_hat = dirichlet_regression.compute_mu_spatial(X, beta_hat, M)

In [36]:
ll_h_moins = dirichlet_regression.dirichlet_loglikelihood(mu_h_moins,reg_spatial.phi,Y)
ll_h_plus = dirichlet_regression.dirichlet_loglikelihood(mu_h_plus,reg_spatial.phi,Y)
ll_hat = dirichlet_regression.dirichlet_loglikelihood(mu_hat,reg_spatial.phi,Y)

In [37]:
(ll_h_plus - ll_hat)/h

0.010488747648196295

In [38]:
(ll_h_plus - 2*ll_hat + ll_h_moins)/(h**2)

-2682.781996554695

In [39]:
np.diag(reg_spatial.hess)

array([-2646.46633003, -2550.83614106,  -524.89835983,  -364.06579574,
        -661.5944185 ,  -615.30963476,  -254.59498532,   -83.58066802,
       -3079.96842806])

In [40]:
print(pd.DataFrame(reg_spatial.hess))

             0            1           2           3           4           5  \
0 -2646.466330  1257.205683 -455.426726   31.487658 -150.799425 -273.866773   
1  1257.205683 -2550.836141   31.487658  223.582910 -273.866773   82.794093   
2  -455.426726    31.487658 -524.898360  164.381209  -89.309285  -32.935222   
3    31.487658   223.582910  164.381209 -364.065796  -32.935222  108.952368   
4  -150.799425  -273.866773  -89.309285  -32.935222 -661.594418  311.678181   
5  -273.866773    82.794093  -32.935222  108.952368  311.678181 -615.309635   
6   -35.332941    73.463759  113.131791 -138.354922    4.816091 -111.949471   
7   -29.350036    35.776830   47.126481  -61.441187    6.161220  -64.101902   
8   569.663349  -688.557175 -215.542938  170.857686   65.686319  343.905402   

            6           7            8  
0  -35.332941  -29.350036   569.663349  
1   73.463759   35.776830  -688.557175  
2  113.131791   47.126481  -215.542938  
3 -138.354922  -61.441187   170.857686  
4   

In [176]:
def compute_reduced_hessian(beta_hat, h=1e-6):
    n_rows, n_cols = beta_hat.shape
    
    # Create mask: True for parameters we want to include in Hessian
    # Exclude first column (index 0)
    mask = np.ones(beta_hat.shape, dtype=bool)
    mask[:, 0] = False  # Exclude first column
    
    # Get indices of active parameters
    active_indices = np.where(mask.flatten())[0]
    n_active = len(active_indices)
    
    # Map from flat index to matrix position for active parameters only
    def get_matrix_pos(active_idx):
        flat_idx = active_indices[active_idx]
        row = flat_idx // n_cols
        col = flat_idx % n_cols
        return row, col
    
    hessian_reduced = np.zeros((n_active, n_active))
    
    # Your original unperturbed log-likelihood
    mu_hat = dirichlet_regression.compute_mu_spatial(X, beta_hat, M)
    ll = dirichlet_regression.dirichlet_loglikelihood(mu_hat,reg_spatial.phi,Y)
    
    for i in range(n_active):
        i_row, i_col = get_matrix_pos(i)
        
        # Diagonal term
        beta_plus = np.copy(beta_hat)
        beta_minus = np.copy(beta_hat)
        beta_plus[i_row, i_col] += h
        beta_minus[i_row, i_col] -= h
        
        mu_plus = dirichlet_regression.compute_mu_spatial(X, beta_plus, M)
        mu_minus = dirichlet_regression.compute_mu_spatial(X, beta_minus, M)
        
        ll_plus = dirichlet_regression.dirichlet_loglikelihood(mu_plus,reg_spatial.phi,Y)
        ll_minus = dirichlet_regression.dirichlet_loglikelihood(mu_minus,reg_spatial.phi,Y)
        
        hessian_reduced[i, i] = (ll_plus - 2*ll + ll_minus) / (h**2)
        
        # Cross terms
        for j in range(i+1, n_active):
            j_row, j_col = get_matrix_pos(j)
            
            # Four perturbations
            beta_pp = np.copy(beta_hat)
            beta_pm = np.copy(beta_hat)
            beta_mp = np.copy(beta_hat)
            beta_mm = np.copy(beta_hat)
            
            beta_pp[i_row, i_col] += h; beta_pp[j_row, j_col] += h
            beta_pm[i_row, i_col] += h; beta_pm[j_row, j_col] -= h
            beta_mp[i_row, i_col] -= h; beta_mp[j_row, j_col] += h
            beta_mm[i_row, i_col] -= h; beta_mm[j_row, j_col] -= h
            
            # Compute mu
            mu_pp = dirichlet_regression.compute_mu_spatial(X, beta_pp, M)
            mu_pm = dirichlet_regression.compute_mu_spatial(X, beta_pm, M)
            mu_mp = dirichlet_regression.compute_mu_spatial(X, beta_mp, M)
            mu_mm = dirichlet_regression.compute_mu_spatial(X, beta_mm, M)
            
            # Compute function values
            ll_pp = dirichlet_regression.dirichlet_loglikelihood(mu_pp,reg_spatial.phi,Y)
            ll_pm = dirichlet_regression.dirichlet_loglikelihood(mu_pm,reg_spatial.phi,Y)
            ll_mp = dirichlet_regression.dirichlet_loglikelihood(mu_mp,reg_spatial.phi,Y)
            ll_mm = dirichlet_regression.dirichlet_loglikelihood(mu_mm,reg_spatial.phi,Y)
            
            hessian_cross = (ll_pp - ll_pm - ll_mp + ll_mm) / (4 * h**2)
            hessian_reduced[i, j] = hessian_cross
            hessian_reduced[j, i] = hessian_cross
    
    return hessian_reduced, active_indices

In [185]:
hess_numeric=compute_reduced_hessian(beta_hat, h=1e-6)

In [186]:
print(pd.DataFrame(hess_numeric[0]))

             0            1           2           3           4           5
0 -2646.174835  1257.234317 -455.486315   31.462832 -150.777169 -273.843170
1  1257.234317 -2550.450517   31.434411  223.593588 -273.843170   82.764018
2  -455.486315    31.434411 -524.778443  164.419589  -89.301011  -32.912340
3    31.462832   223.593588  164.419589 -363.911568  -32.940761  108.968834
4  -150.777169  -273.843170  -89.301011  -32.940761 -661.202648  311.644044
5  -273.843170    82.764018  -32.912340  108.968834  311.644044 -614.932105


In [192]:
print(pd.DataFrame(reg_spatial.hess))

             0            1           2           3           4           5  \
0 -2646.466330  1257.205683 -455.426726   31.487658 -150.799425 -273.866773   
1  1257.205683 -2550.836141   31.487658  223.582910 -273.866773   82.794093   
2  -455.426726    31.487658 -524.898360  164.381209  -89.309285  -32.935222   
3    31.487658   223.582910  164.381209 -364.065796  -32.935222  108.952368   
4  -150.799425  -273.866773  -89.309285  -32.935222 -661.594418  311.678181   
5  -273.866773    82.794093  -32.935222  108.952368  311.678181 -615.309635   
6   -17.416401    36.211938   98.458259 -125.125711    4.034087 -103.420507   
7   -14.467293    17.635204   39.256001  -55.134801    0.313167  -53.782362   
8   569.663349  -688.557175 -215.542938  170.857686   65.686319  343.905402   

            6           7            8  
0  -17.416401  -14.467293   569.663349  
1   36.211938   17.635204  -688.557175  
2   98.458259   39.256001  -215.542938  
3 -125.125711  -55.134801   170.857686  
4   

In [114]:
#h = np.sqrt(sys.float_info.epsilon)
h = 10e-7

In [115]:
zeros_col = np.zeros((reg_spatial.beta.shape[0], 1))
beta_hat = np.hstack([zeros_col, reg_spatial.beta])

In [44]:
gamma_hat = reg_spatial.gamma

In [122]:
gamma_h_plus = np.copy(gamma_hat)
gamma_h_plus[1] += h
gamma_h_moins = np.copy(gamma_hat)
gamma_h_moins[1] -= h

In [123]:
phi_h_plus = np.exp(np.matmul(Z,gamma_h_plus))
phi_h_moins = np.exp(np.matmul(Z,gamma_h_moins))

In [124]:
ll_h_moins = dirichlet_regression.dirichlet_loglikelihood(mu_hat,phi_h_moins,Y)
ll_h_plus = dirichlet_regression.dirichlet_loglikelihood(mu_hat,phi_h_plus,Y)
ll_hat = dirichlet_regression.dirichlet_loglikelihood(mu_hat,reg_spatial.phi,Y)

In [125]:
(ll_h_plus - ll)/h

-0.0012882992450613528

In [126]:
(ll_h_plus - 2*ll + ll_h_moins)/(h**2)

-83.44613888766617

In [41]:
def compute_hessian_gamma_and_cross(beta_hat, gamma_hat, X, Z, M, Y, h=1e-6):
    # ------------------------------
    # Unperturbed quantities
    # ------------------------------
    mu_hat = dirichlet_regression.compute_mu_spatial(X, beta_hat, M)
    phi_hat = np.exp(Z @ gamma_hat)
    ll_hat = dirichlet_regression.dirichlet_loglikelihood(mu_hat, phi_hat, Y)

    # ------------------------------
    # Dimensions
    # ------------------------------
    n_rows, n_cols = beta_hat.shape
    n_beta = n_rows * n_cols       # flattened beta
    G = gamma_hat.size

    # Allocate Hessians
    H_gamma = np.zeros((G, G))         # gamma-gamma block
    H_cross = np.zeros((n_beta, G))    # beta-gamma cross block

    # =====================================================
    # 1. Gamma–Gamma Hessian Block
    # =====================================================
    for i in range(G):

        # diagonal term
        gamma_plus = gamma_hat.copy()
        gamma_minus = gamma_hat.copy()
        gamma_plus[i] += h
        gamma_minus[i] -= h

        phi_plus = np.exp(Z @ gamma_plus)
        phi_minus = np.exp(Z @ gamma_minus)

        ll_plus = dirichlet_regression.dirichlet_loglikelihood(mu_hat, phi_plus, Y)
        ll_minus = dirichlet_regression.dirichlet_loglikelihood(mu_hat, phi_minus, Y)

        H_gamma[i, i] = (ll_plus - 2*ll_hat + ll_minus) / (h**2)

        # off-diagonal gamma-gamma
        for j in range(i+1, G):

            g_pp = gamma_hat.copy()
            g_pm = gamma_hat.copy()
            g_mp = gamma_hat.copy()
            g_mm = gamma_hat.copy()

            g_pp[i] += h; g_pp[j] += h
            g_pm[i] += h; g_pm[j] -= h
            g_mp[i] -= h; g_mp[j] += h
            g_mm[i] -= h; g_mm[j] -= h

            ll_pp = dirichlet_regression.dirichlet_loglikelihood(mu_hat, np.exp(Z @ g_pp), Y)
            ll_pm = dirichlet_regression.dirichlet_loglikelihood(mu_hat, np.exp(Z @ g_pm), Y)
            ll_mp = dirichlet_regression.dirichlet_loglikelihood(mu_hat, np.exp(Z @ g_mp), Y)
            ll_mm = dirichlet_regression.dirichlet_loglikelihood(mu_hat, np.exp(Z @ g_mm), Y)

            h_ij = (ll_pp - ll_pm - ll_mp + ll_mm) / (4 * h**2)
            H_gamma[i, j] = h_ij
            H_gamma[j, i] = h_ij

    # =====================================================
    # 2. Beta–Gamma Cross Hessian Block
    # =====================================================
    for b in range(n_beta):

        b_row = b // n_cols
        b_col = b % n_cols

        for g in range(G):

            # Four perturbed configurations
            beta_pp = beta_hat.copy()
            beta_pm = beta_hat.copy()
            beta_mp = beta_hat.copy()
            beta_mm = beta_hat.copy()

            gamma_pp = gamma_hat.copy()
            gamma_pm = gamma_hat.copy()
            gamma_mp = gamma_hat.copy()
            gamma_mm = gamma_hat.copy()

            # Apply perturbations
            beta_pp[b_row, b_col] += h
            beta_pm[b_row, b_col] += h
            beta_mp[b_row, b_col] -= h
            beta_mm[b_row, b_col] -= h

            gamma_pp[g] += h
            gamma_pm[g] -= h
            gamma_mp[g] += h
            gamma_mm[g] -= h

            # Evaluate each of the four perturbations
            ll_pp = dirichlet_regression.dirichlet_loglikelihood(
                dirichlet_regression.compute_mu_spatial(X, beta_pp, M),
                np.exp(Z @ gamma_pp),
                Y
            )

            ll_pm = dirichlet_regression.dirichlet_loglikelihood(
                dirichlet_regression.compute_mu_spatial(X, beta_pp, M),
                np.exp(Z @ gamma_pm),
                Y
            )

            ll_mp = dirichlet_regression.dirichlet_loglikelihood(
                dirichlet_regression.compute_mu_spatial(X, beta_mp, M),
                np.exp(Z @ gamma_mp),
                Y
            )

            ll_mm = dirichlet_regression.dirichlet_loglikelihood(
                dirichlet_regression.compute_mu_spatial(X, beta_mp, M),
                np.exp(Z @ gamma_mm),
                Y
            )

            H_cross[b, g] = (ll_pp - ll_pm - ll_mp + ll_mm) / (4 * h**2)

    return H_gamma, H_cross

In [45]:
H_gamma, H_cross = compute_hessian_gamma_and_cross(beta_hat, gamma_hat, X, Z, M, Y, h=1e-6)

In [46]:
H_gamma

array([[-254.43114282, -125.76606423],
       [-125.76606423,  -83.44613889]])

In [49]:
H_cross

array([[ -38.14193406,   -6.42330633],
       [ -35.35660653,  -29.35962584],
       [  73.44169717,   35.75451046],
       [  25.18163456,   14.35296326],
       [ 113.17524695,   47.15161595],
       [-138.38530322,  -61.41931408],
       [ 107.14984455,   57.95186553],
       [   4.88853402,    6.16751095],
       [-111.92469174,  -64.00568964]])

In [48]:
pd.DataFrame(reg_spatial.hess)

,0,1,2,3,4,5,6,7,8
0,-2646.466330,1257.205683,-455.426726,31.487658,-150.799425,-273.866773,-35.332941,-29.350036,569.663349
1,1257.205683,-2550.836141,31.487658,223.582910,-273.866773,82.794093,73.463759,35.776830,-688.557175
2,-455.426726,31.487658,-524.898360,164.381209,-89.309285,-32.935222,113.131791,47.126481,-215.542938
3,31.487658,223.582910,164.381209,-364.065796,-32.935222,108.952368,-138.354922,-61.441187,170.857686
4,-150.799425,-273.866773,-89.309285,-32.935222,-661.594418,311.678181,4.816091,6.161220,65.686319
5,-273.866773,82.794093,-32.935222,108.952368,311.678181,-615.309635,-111.949471,-64.101902,343.905402
6,-35.332941,73.463759,113.131791,-138.354922,4.816091,-111.949471,-254.594985,-125.725359,209.647446
7,-29.350036,35.776830,47.126481,-61.441187,6.161220,-64.101902,-125.725359,-83.580668,114.950330
8,569.663349,-688.557175,-215.542938,170.857686,65.686319,343.905402,209.647446,114.950330,-3079.968428


In [127]:
#h = np.sqrt(sys.float_info.epsilon)
h = 10e-7

In [372]:
zeros_col = np.zeros((reg_spatial.beta.shape[0], 1))
beta_hat = np.hstack([zeros_col, reg_spatial.beta])

In [373]:
beta_h_plus = np.copy(beta_hat)
beta_h_plus[0,1] += h
beta_h_moins = np.copy(beta_hat)
beta_h_moins[0,1] -= h

In [374]:
mu_h_moins = dirichlet_regression.compute_mu_spatial(X, beta_h_moins, M)
mu_h_plus = dirichlet_regression.compute_mu_spatial(X, beta_h_plus, M)
mu_hat = dirichlet_regression.compute_mu_spatial(X, beta_hat, M)

In [375]:
ll_h_moins = dirichlet_regression.dirichlet_loglikelihood(mu_h_moins,reg_spatial.phi,Y)
ll_h_plus = dirichlet_regression.dirichlet_loglikelihood(mu_h_plus,reg_spatial.phi,Y)
ll_hat = dirichlet_regression.dirichlet_loglikelihood(mu_hat,reg_spatial.phi,Y)

In [376]:
(ll_h_plus - ll)/h

-0.0020816059986827895

In [377]:
(ll_h_plus - 2*ll + ll_h_moins)/(h**2)

-2287.1518012834713

In [370]:
reg_not_spatial.gamma

array([1.63364338, 0.19344572])

In [302]:
%%time
reg_not_spatial = dirichlet_regression.dirichletRegressor(spatial=False, maxfun=5000)
reg_not_spatial.fit(X, Y, parametrization='alternative', Z=Z, fit_intercept=False, verbose=1)

Optimization terminated successfully.
Wall time: 15.6 ms


In [325]:
h = 10e-6

In [326]:
dirichlet_regression.dirichlet_loglikelihood(reg_not_spatial.mu,reg_not_spatial.phi,Y)

577.5758545935569

In [327]:
zeros_col = np.zeros((reg_not_spatial.beta.shape[0], 1))
beta_hat = np.hstack([zeros_col, reg_not_spatial.beta])

In [328]:
beta_hat

array([[ 0.        ,  0.1818285 ,  0.45968187],
       [ 0.        ,  0.72898001, -1.09582795],
       [ 0.        , -0.76096164, -1.20606401]])

In [329]:
beta_h_plus = np.copy(beta_hat)
beta_h_plus[0,1] += h
beta_h_moins = np.copy(beta_hat)
beta_h_moins[0,1] -= h

In [330]:
mu_h_moins = dirichlet_regression.compute_mu(X, beta_h_moins)
mu_h_plus = dirichlet_regression.compute_mu(X, beta_h_plus)
mu_hat = dirichlet_regression.compute_mu(X, beta_hat)

In [331]:
ll_h_moins = dirichlet_regression.dirichlet_loglikelihood(mu_h_moins,reg_not_spatial.phi,Y)
ll_h_plus = dirichlet_regression.dirichlet_loglikelihood(mu_h_plus,reg_not_spatial.phi,Y)
ll_hat = dirichlet_regression.dirichlet_loglikelihood(mu_hat,reg_not_spatial.phi,Y)

In [332]:
(ll_h_plus - ll_hat)/h

-0.0012100599633413367

In [333]:
(ll_h_plus - 2*ll_hat + ll_h_moins)/(h**2)

-311.67019187705586

In [334]:
reg_not_spatial.inference(X, Y, Z, display=True)

-----
Estimated parameter beta_11 = 0.18, se = 0.07, CI 95% = [0.04 ; 0.33],  p-value = 0.01
-----
Estimated parameter beta_12 = 0.46, se = 0.07, CI 95% = [0.33 ; 0.59],  p-value = 0.0
-----
Estimated parameter beta_21 = 0.73, se = 0.08, CI 95% = [0.57 ; 0.89],  p-value = 0.0
-----
Estimated parameter beta_22 = -1.1, se = 0.08, CI 95% = [-1.26 ; -0.93],  p-value = 0.0
-----
Estimated parameter beta_31 = -0.76, se = 0.08, CI 95% = [-0.92 ; -0.61],  p-value = 0.0
-----
Estimated parameter beta_32 = -1.21, se = 0.08, CI 95% = [-1.37 ; -1.04],  p-value = 0.0
-----
Estimated parameter gamma_1 = 1.63, se = 0.14, CI 95% = [1.36 ; 1.91],  p-value = 0.0
-----
Estimated parameter gamma_2 = 0.19, se = 0.21, CI 95% = [-0.22 ; 0.61],  p-value = 0.36


In [335]:
np.diag(reg_not_spatial.hess)

array([-311.67143727, -339.35610572, -267.10974432, -249.40670932,
       -275.60060298, -274.13400054, -279.02618467, -100.41890675])

In [79]:
%%time
reg_spatial_ce = dirichlet_regression.dirichletRegressor(spatial=True)
reg_spatial_ce.fit(X, Y, parametrization='alternative', Z=Z, W=W, fit_intercept=False, verbose=1, loss='crossentropy')


Optimization terminated successfully.
Wall time: 606 ms


Test on another set of data (rerun the generation of X and W beforehand)

In [81]:
mu_pred = reg_spatial.pred(X,W)
print('R2:',r2_score(Y,mu_pred))
print('Cos similarity:',cos_similarity(Y, mu_pred))
print('RMSE:',mean_squared_error(Y,mu_pred,squared=False))

R2: 0.7406959828864667
Cos similarity: 0.9779101601507182
RMSE: 0.1225223647569414


In [82]:
mu_pred = reg_spatial_ce.pred(X,W)
print('R2:',r2_score(Y,mu_pred))
print('Cos similarity:',cos_similarity(Y, mu_pred))
print('RMSE:',mean_squared_error(Y,mu_pred,squared=False))

R2: 0.9594509374281396
Cos similarity: 0.9968907893785736
RMSE: 0.04279544323423783


---

In [232]:
def estimating_parameters(rho, n_repeat=100, list_n_samples=[50,200,1000], cov_matrix=None):
    list_solutions_spatial, list_solutions_no_spatial = [], []
    list_solutions_ce_spatial, list_solutions_ce_no_spatial = [], []

    seed=0

    for i in range(len(list_n_samples)):
        n_samples = list_n_samples[i]

        true_params = np.concatenate([beta.flatten(),gamma_var, [rho]])

        solutions_spatial_temp, solutions_no_spatial_temp = [], []
        solutions_ce_spatial_temp, solutions_ce_no_spatial_temp = [], []
        for _ in range(n_repeat):
            np.random.seed(seed)

            X,Z,W = create_features_matrices(n_samples,n_features,choice_W='random_distance',nneighbors=10,cov_mat=cov_matrix)
            Z[:,0] = 1
            M = np.identity(n_samples) - rho*W
            
            try:
                mu = dirichlet_regression.compute_mu_spatial(X, beta, M)
                phi = np.exp(np.matmul(Z,gamma_var))
                alpha = mu*phi[:,None]

                Y = np.array([np.random.dirichlet(alpha_i) for alpha_i in alpha])
                Y = (Y*(n_samples-1)+1/n_classes)/n_samples

                reg_spatial = dirichlet_regression.dirichletRegressor(spatial=True, maxfun=5000)
                reg_spatial.fit(X, Y, parametrization='alternative', Z=Z, W=W, fit_intercept=False, verbose=0)
                solutions_spatial_temp.append(np.concatenate([reg_spatial.beta.flatten(),reg_spatial.gamma,[reg_spatial.rho]]))

                reg_no_spatial = dirichlet_regression.dirichletRegressor(spatial=False, maxfun=5000)
                reg_no_spatial.fit(X, Y, parametrization='alternative', Z=Z, fit_intercept=False, verbose=0)
                solutions_no_spatial_temp.append(np.concatenate([reg_no_spatial.beta.flatten(),reg_no_spatial.gamma]))

                reg_spatial_ce = dirichlet_regression.dirichletRegressor(spatial=True, maxfun=5000)
                reg_spatial_ce.fit(X, Y, loss='crossentropy', W=W, fit_intercept=False, verbose=0)
                solutions_ce_spatial_temp.append(np.concatenate([reg_spatial_ce.beta.flatten(),[reg_spatial_ce.rho]]))

                reg_no_spatial_ce = dirichlet_regression.dirichletRegressor(spatial=False, maxfun=5000)
                reg_no_spatial_ce.fit(X, Y, loss='crossentropy', fit_intercept=False, verbose=0)
                solutions_ce_no_spatial_temp.append(reg_no_spatial_ce.beta.flatten())

            except RuntimeError:
                print("Factor is exactly singular")
            except np.linalg.LinAlgError:
                print("Singular matrix")

            seed+=1
        list_solutions_spatial.append(solutions_spatial_temp)
        list_solutions_no_spatial.append(solutions_no_spatial_temp)
        list_solutions_ce_spatial.append(solutions_ce_spatial_temp)
        list_solutions_ce_no_spatial.append(solutions_ce_no_spatial_temp)
    return(list_solutions_spatial, list_solutions_no_spatial, list_solutions_ce_spatial, list_solutions_ce_no_spatial)

## rho=0.1

In [233]:
%%time
list_solutions_spatial, list_solutions_no_spatial, list_solutions_ce_spatial, list_solutions_ce_no_spatial = estimating_parameters(0.1, n_repeat=100, list_n_samples=[50,200,1000], cov_matrix=cov_matrix)

Wall time: 17min 22s


In [234]:
true_params_effective = np.concatenate([beta[:,1:].flatten(),gamma_var, [rho]])
true_params_effective_no_spatial = np.concatenate([beta[:,1:].flatten(),gamma_var])

In [235]:
np.save('Data Dirichlet/dirichlet_solutions_spatial_rho01.npy',list_solutions_spatial)
np.save('Data Dirichlet/dirichlet_solutions_no_spatial_rho01.npy',list_solutions_no_spatial)
np.save('Data Dirichlet/dirichlet_solutions_ce_spatial_rho01.npy',list_solutions_ce_spatial)
np.save('Data Dirichlet/dirichlet_solutions_ce_no_spatial_rho01.npy',list_solutions_ce_no_spatial)

## rho=0.5

In [242]:
%%time
list_solutions_spatial, list_solutions_no_spatial, list_solutions_ce_spatial, list_solutions_ce_no_spatial = estimating_parameters(0.5, n_repeat=100, list_n_samples=[50,200,1000], cov_matrix=cov_matrix)

Singular matrix
Singular matrix
Singular matrix
Wall time: 29min 54s


In [243]:
np.save('Data Dirichlet/dirichlet_solutions_spatial_rho05.npy',list_solutions_spatial)
np.save('Data Dirichlet/dirichlet_solutions_no_spatial_rho05.npy',list_solutions_no_spatial)
np.save('Data Dirichlet/dirichlet_solutions_ce_spatial_rho05.npy',list_solutions_ce_spatial)
np.save('Data Dirichlet/dirichlet_solutions_ce_no_spatial_rho05.npy',list_solutions_ce_no_spatial)

C:\Users\tnguyen001\AppData\Roaming\Python\Python38\site-packages\numpy\core\_asarray.py:136: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  return array(a, dtype, copy=False, order=order, subok=True)


## rho = 0.9

In [251]:
%%time
list_solutions_spatial, list_solutions_no_spatial, list_solutions_ce_spatial, list_solutions_ce_no_spatial = estimating_parameters(0.9, n_repeat=100, list_n_samples=[50,200,1000], cov_matrix=cov_matrix)

Singular matrix
Singular matrix
Singular matrix


C:\Users\tnguyen001\AppData\Roaming\Python\Python38\site-packages\scipy\optimize\optimize.py:697: RuntimeWarning: invalid value encountered in double_scalars
  df = (f(*((xk + d,) + args)) - f0) / d[k]
C:\Users\tnguyen001\AppData\Roaming\Python\Python38\site-packages\scipy\optimize\optimize.py:697: RuntimeWarning: invalid value encountered in double_scalars
  df = (f(*((xk + d,) + args)) - f0) / d[k]


Wall time: 1h 13min 6s


In [252]:
np.save('Data Dirichlet/dirichlet_solutions_spatial_rho09.npy',list_solutions_spatial)
np.save('Data Dirichlet/dirichlet_solutions_no_spatial_rho09.npy',list_solutions_no_spatial)
np.save('Data Dirichlet/dirichlet_solutions_ce_spatial_rho09.npy',list_solutions_ce_spatial)
np.save('Data Dirichlet/dirichlet_solutions_ce_no_spatial_rho09.npy',list_solutions_ce_no_spatial)

C:\Users\tnguyen001\AppData\Roaming\Python\Python38\site-packages\numpy\core\_asarray.py:136: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  return array(a, dtype, copy=False, order=order, subok=True)


## rho=0

In [276]:
%%time
list_solutions_spatial, list_solutions_no_spatial, list_solutions_ce_spatial, list_solutions_ce_no_spatial = estimating_parameters(0., n_repeat=100, list_n_samples=[50,200,1000], cov_matrix=cov_matrix)

Wall time: 22min 27s


In [277]:
np.save('Data Dirichlet/dirichlet_solutions_spatial_rho00.npy',list_solutions_spatial)
np.save('Data Dirichlet/dirichlet_solutions_no_spatial_rho00.npy',list_solutions_no_spatial)
np.save('Data Dirichlet/dirichlet_solutions_ce_spatial_rho00.npy',list_solutions_ce_spatial)
np.save('Data Dirichlet/dirichlet_solutions_ce_no_spatial_rho00.npy',list_solutions_ce_no_spatial)

# Analysis of the results

In [9]:
from texttable import Texttable
import latextable

In [10]:
list_solutions_spatial_01 = np.load('Data Dirichlet/dirichlet_solutions_spatial_rho01.npy')
list_solutions_no_spatial_01 = np.load('Data Dirichlet/dirichlet_solutions_no_spatial_rho01.npy')
list_solutions_ce_spatial_01 = np.load('Data Dirichlet/dirichlet_solutions_ce_spatial_rho01.npy')
list_solutions_ce_no_spatial_01 = np.load('Data Dirichlet/dirichlet_solutions_ce_no_spatial_rho01.npy')

In [11]:
list_solutions_spatial_05 = np.load('Data Dirichlet/dirichlet_solutions_spatial_rho05.npy', allow_pickle=True)
list_solutions_no_spatial_05 = np.load('Data Dirichlet/dirichlet_solutions_no_spatial_rho05.npy', allow_pickle=True)
list_solutions_ce_spatial_05 = np.load('Data Dirichlet/dirichlet_solutions_ce_spatial_rho05.npy', allow_pickle=True)
list_solutions_ce_no_spatial_05 = np.load('Data Dirichlet/dirichlet_solutions_ce_no_spatial_rho05.npy', allow_pickle=True)

In [12]:
list_solutions_spatial_09 = np.load('Data Dirichlet/dirichlet_solutions_spatial_rho09.npy', allow_pickle=True)
list_solutions_no_spatial_09 = np.load('Data Dirichlet/dirichlet_solutions_no_spatial_rho09.npy', allow_pickle=True)
list_solutions_ce_spatial_09 = np.load('Data Dirichlet/dirichlet_solutions_ce_spatial_rho09.npy', allow_pickle=True)
list_solutions_ce_no_spatial_09 = np.load('Data Dirichlet/dirichlet_solutions_ce_no_spatial_rho09.npy', allow_pickle=True)

In [13]:
true_params_effective_01 = np.concatenate([beta[:,1:].flatten(),gamma_var, [0.1]])
true_params_effective_05 = np.concatenate([beta[:,1:].flatten(),gamma_var, [0.5]])
true_params_effective_09 = np.concatenate([beta[:,1:].flatten(),gamma_var, [0.9]])
true_params_effective_no_spatial = np.concatenate([beta[:,1:].flatten(),gamma_var])

In [14]:
list_solutions_spatial_00 = np.load('Data Dirichlet/dirichlet_solutions_spatial_rho00.npy')
list_solutions_no_spatial_00 = np.load('Data Dirichlet/dirichlet_solutions_no_spatial_rho00.npy')
list_solutions_ce_spatial_00 = np.load('Data Dirichlet/dirichlet_solutions_ce_spatial_rho00.npy', allow_pickle=True)
list_solutions_ce_no_spatial_00 = np.load('Data Dirichlet/dirichlet_solutions_ce_no_spatial_rho00.npy', allow_pickle=True)
true_params_effective_00 = np.concatenate([beta[:,1:].flatten(),gamma_var, [0.]])

### Results of maximum likelihood

In [281]:
table_params = Texttable()
table_params.add_row(["Parameter", "S n=50", "S n=200", "S n=1000",
                      "NS n=50", "NS n=200", "NS n=1000"])
                      
#param_names = ["$\\beta_{01}$", "", "$\\beta_{02}$", "", "$\\beta_{11}$", "", "$\\beta_{12}$", "",
#               "$\\beta_{21}$", "", "$\\beta_{22}$", "", "$\\gamma_0$", "", "$\\gamma_1$", "", "$\\rho$", ""]

param_names = ["\\multirow{2}{*}{$\\beta_{01}$}", "\\multirow{2}{*}{$\\beta_{02}$}", "\\multirow{2}{*}{$\\beta_{11}$}",
               "\\multirow{2}{*}{$\\beta_{12}$}", "\\multirow{2}{*}{$\\beta_{21}$}", "\\multirow{2}{*}{$\\beta_{22}$}",
               "\\multirow{2}{*}{$\\gamma_0$}", "\\multirow{2}{*}{$\\gamma_1$}", "\\multirow{2}{*}{$\\rho$}" ]
param_names_2 = [""]*len(param_names)

columns_1 = []
columns_2 = []

for ns in range(3):
    col_to_add = [str(np.round(np.mean(list_solutions_spatial_01[ns][:,i] - true_params_effective_01[i]),3))
                  +' ('+str(np.round(np.std(list_solutions_spatial_01[ns][:,i] - true_params_effective_01[i]),3))+')' for i in range(len(true_params_effective_01))]
    columns_1.append(col_to_add)
    col_to_add = ['['+  str(np.round(mean_squared_error(list_solutions_spatial_01[ns][:,i], [true_params_effective_01[i]]*100),3)) +']' for i in range(len(true_params_effective_01))]
    columns_2.append(col_to_add)
    
for ns in range(3):
    col_to_add = [str(np.round(np.mean(list_solutions_no_spatial_01[ns][:,i] - true_params_effective_no_spatial[i]),3))
                  +' ('+str(np.round(np.std(list_solutions_no_spatial_01[ns][:,i] - true_params_effective_no_spatial[i]),3))+')' for i in range(len(true_params_effective_no_spatial))] + ["/"]
    columns_1.append(col_to_add)
    col_to_add = [' ['+  str(np.round(mean_squared_error(list_solutions_no_spatial_01[ns][:,i], [true_params_effective_no_spatial[i]]*100),3)) +']' for i in range(len(true_params_effective_no_spatial))] + ["/"]
    columns_2.append(col_to_add)
        
columns_1.insert(0,param_names)
columns_2.insert(0,param_names_2)

tr_columns_1 = np.transpose(columns_1)
tr_columns_2 = np.transpose(columns_2)
for i in range(len(tr_columns_1)):
    table_params.add_row(tr_columns_1[i])
    table_params.add_row(tr_columns_2[i])

print(table_params.draw())

+-----------+-----------+----------+----------+----------+----------+----------+
| Parameter | S n=50    | S n=200  | S n=1000 | NS n=50  | NS n=200 | NS       |
|           |           |          |          |          |          | n=1000   |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | 0.013     | 0.012    | 0.01     | 0.014    | 0.012    | 0.011    |
| {2}{*}{$\ | (0.059)   | (0.032)  | (0.016)  | (0.067)  | (0.036)  | (0.018)  |
| beta_{01} |           |          |          |          |          |          |
| $}        |           |          |          |          |          |          |
+-----------+-----------+----------+----------+----------+----------+----------+
|           | [0.004]   | [0.001]  | [0.0]    |  [0.005] |  [0.001] |  [0.0]   |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | 0.038     | 0.035    | 0.031    | 0.051    | 0.055    | 0.047    |
| {2}{*}{$\ | (0.087)   | (0

In [262]:
print(latextable.draw_latex(table_params, caption="Difference between the estimated parameters and the true parameters ($rho=0.1$). The results are presented as the mean of the differences on the 100 iterations, the standard deviation within parenthesis, and the standard deviation within square brackets."))

\begin{table}
	\begin{center}
		\begin{tabular}{|l|l|l|l|l|l|l|}
			\hline
			 \\
			\hline
			Parameter & S n=50 & S n=200 & S n=1000 & NS n=50 & NS n=200 & NS n=1000 \\
			\hline
			\multirow{2}{*}{$\beta_{01}$} & 0.013 (0.059) & 0.012 (0.032) & 0.01 (0.016) & 0.014 (0.067) & 0.012 (0.036) & 0.011 (0.018) \\
			\hline
			 & [0.004] & [0.001] & [0.0] &  [0.005] &  [0.001] &  [0.0] \\
			\hline
			\multirow{2}{*}{$\beta_{02}$} & 0.038 (0.087) & 0.035 (0.034) & 0.031 (0.016) & 0.051 (0.106) & 0.055 (0.046) & 0.047 (0.021) \\
			\hline
			 & [0.009] & [0.002] & [0.001] &  [0.014] &  [0.005] &  [0.003] \\
			\hline
			\multirow{2}{*}{$\beta_{11}$} & -0.062 (0.079) & -0.037 (0.038) & -0.019 (0.017) & -0.067 (0.081) & -0.04 (0.038) & -0.021 (0.017) \\
			\hline
			 & [0.01] & [0.003] & [0.001] &  [0.011] &  [0.003] &  [0.001] \\
			\hline
			\multirow{2}{*}{$\beta_{12}$} & 0.277 (0.12) & 0.185 (0.051) & 0.141 (0.023) & 0.288 (0.127) & 0.198 (0.055) & 0.158 (0.024) \\
			\hline
			 & [0.091]

In [272]:
table_params = Texttable()
table_params.add_row(["Parameter", "S n=50", "S n=200", "S n=1000",
                      "NS n=50", "NS n=200", "NS n=1000"])
                      
#param_names = ["$\\beta_{01}$", "", "$\\beta_{02}$", "", "$\\beta_{11}$", "", "$\\beta_{12}$", "",
#               "$\\beta_{21}$", "", "$\\beta_{22}$", "", "$\\gamma_0$", "", "$\\gamma_1$", "", "$\\rho$", ""]

param_names = ["\\multirow{2}{*}{$\\beta_{01}$}", "\\multirow{2}{*}{$\\beta_{02}$}", "\\multirow{2}{*}{$\\beta_{11}$}",
               "\\multirow{2}{*}{$\\beta_{12}$}", "\\multirow{2}{*}{$\\beta_{21}$}", "\\multirow{2}{*}{$\\beta_{22}$}",
               "\\multirow{2}{*}{$\\gamma_0$}", "\\multirow{2}{*}{$\\gamma_1$}", "\\multirow{2}{*}{$\\rho$}" ]
param_names_2 = [""]*len(param_names)

columns_1 = []
columns_2 = []

for ns in range(3):
    col_to_add = [str(np.round(np.mean(np.array(list_solutions_spatial_05[ns])[:,i] - true_params_effective_05[i]),3))
                  +' ('+str(np.round(np.std(np.array(list_solutions_spatial_05[ns])[:,i] - true_params_effective_05[i]),3))+')' for i in range(len(true_params_effective_05))]
    columns_1.append(col_to_add)
    col_to_add = ['['+  str(np.round(mean_squared_error(np.array(list_solutions_spatial_05[ns])[:,i], [true_params_effective_05[i]]*len(list_solutions_spatial_05[ns])),3)) +']' for i in range(len(true_params_effective_05))]
    columns_2.append(col_to_add)
    
for ns in range(3):
    col_to_add = [str(np.round(np.mean(np.array(list_solutions_no_spatial_05[ns])[:,i] - true_params_effective_no_spatial[i]),3))
                  +' ('+str(np.round(np.std(np.array(list_solutions_no_spatial_05[ns])[:,i] - true_params_effective_no_spatial[i]),3))+')' for i in range(len(true_params_effective_no_spatial))] + ["/"]
    columns_1.append(col_to_add)
    col_to_add = [' ['+  str(np.round(mean_squared_error(np.array(list_solutions_no_spatial_05[ns])[:,i], [true_params_effective_no_spatial[i]]*len(list_solutions_no_spatial_05[ns])),3)) +']' for i in range(len(true_params_effective_no_spatial))] + ["/"]
    columns_2.append(col_to_add)
        
columns_1.insert(0,param_names)
columns_2.insert(0,param_names_2)

tr_columns_1 = np.transpose(columns_1)
tr_columns_2 = np.transpose(columns_2)
for i in range(len(tr_columns_1)):
    table_params.add_row(tr_columns_1[i])
    table_params.add_row(tr_columns_2[i])

print(table_params.draw())

+-----------+-----------+----------+----------+----------+----------+----------+
| Parameter | S n=50    | S n=200  | S n=1000 | NS n=50  | NS n=200 | NS       |
|           |           |          |          |          |          | n=1000   |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | 0.009     | 0.011    | 0.007    | 0.029    | 0.03     | 0.028    |
| {2}{*}{$\ | (0.042)   | (0.02)   | (0.009)  | (0.157)  | (0.079)  | (0.037)  |
| beta_{01} |           |          |          |          |          |          |
| $}        |           |          |          |          |          |          |
+-----------+-----------+----------+----------+----------+----------+----------+
|           | [0.002]   | [0.001]  | [0.0]    |  [0.026] |  [0.007] |  [0.002] |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | 0.012     | 0.023    | 0.017    | 0.116    | 0.184    | 0.144    |
| {2}{*}{$\ | (0.047)   | (0

In [273]:
print(latextable.draw_latex(table_params, caption="Difference between the estimated parameters and the true parameters ($rho=0.5$). The results are presented as the mean of the differences on the 100 iterations, the standard deviation within parenthesis, and the standard deviation within square brackets."))

\begin{table}
	\begin{center}
		\begin{tabular}{|l|l|l|l|l|l|l|}
			\hline
			 \\
			\hline
			Parameter & S n=50 & S n=200 & S n=1000 & NS n=50 & NS n=200 & NS n=1000 \\
			\hline
			\multirow{2}{*}{$\beta_{01}$} & 0.009 (0.042) & 0.011 (0.02) & 0.007 (0.009) & 0.029 (0.157) & 0.03 (0.079) & 0.028 (0.037) \\
			\hline
			 & [0.002] & [0.001] & [0.0] &  [0.026] &  [0.007] &  [0.002] \\
			\hline
			\multirow{2}{*}{$\beta_{02}$} & 0.012 (0.047) & 0.023 (0.017) & 0.017 (0.009) & 0.116 (0.312) & 0.184 (0.147) & 0.144 (0.066) \\
			\hline
			 & [0.002] & [0.001] & [0.0] &  [0.111] &  [0.055] &  [0.025] \\
			\hline
			\multirow{2}{*}{$\beta_{11}$} & -0.06 (0.083) & -0.037 (0.037) & -0.024 (0.016) & -0.163 (0.122) & -0.17 (0.057) & -0.155 (0.027) \\
			\hline
			 & [0.011] & [0.003] & [0.001] &  [0.042] &  [0.032] &  [0.025] \\
			\hline
			\multirow{2}{*}{$\beta_{12}$} & 0.356 (0.123) & 0.221 (0.061) & 0.176 (0.029) & 0.682 (0.205) & 0.622 (0.099) & 0.598 (0.049) \\
			\hline
			 & [0.142]

In [274]:
table_params = Texttable()
table_params.add_row(["Parameter", "S n=50", "S n=200", "S n=1000",
                      "NS n=50", "NS n=200", "NS n=1000"])
                      
#param_names = ["$\\beta_{01}$", "", "$\\beta_{02}$", "", "$\\beta_{11}$", "", "$\\beta_{12}$", "",
#               "$\\beta_{21}$", "", "$\\beta_{22}$", "", "$\\gamma_0$", "", "$\\gamma_1$", "", "$\\rho$", ""]

param_names = ["\\multirow{2}{*}{$\\beta_{01}$}", "\\multirow{2}{*}{$\\beta_{02}$}", "\\multirow{2}{*}{$\\beta_{11}$}",
               "\\multirow{2}{*}{$\\beta_{12}$}", "\\multirow{2}{*}{$\\beta_{21}$}", "\\multirow{2}{*}{$\\beta_{22}$}",
               "\\multirow{2}{*}{$\\gamma_0$}", "\\multirow{2}{*}{$\\gamma_1$}", "\\multirow{2}{*}{$\\rho$}" ]
param_names_2 = [""]*len(param_names)

columns_1 = []
columns_2 = []

for ns in range(3):
    col_to_add = [str(np.round(np.mean(np.array(list_solutions_spatial_09[ns])[:,i] - true_params_effective_09[i]),3))
                  +' ('+str(np.round(np.std(np.array(list_solutions_spatial_09[ns])[:,i] - true_params_effective_09[i]),3))+')' for i in range(len(true_params_effective_09))]
    columns_1.append(col_to_add)
    col_to_add = ['['+  str(np.round(mean_squared_error(np.array(list_solutions_spatial_09[ns])[:,i], [true_params_effective_09[i]]*len(list_solutions_spatial_09[ns])),3)) +']' for i in range(len(true_params_effective_09))]
    columns_2.append(col_to_add)
    
for ns in range(3):
    col_to_add = [str(np.round(np.mean(np.array(list_solutions_no_spatial_09[ns])[:,i] - true_params_effective_no_spatial[i]),3))
                  +' ('+str(np.round(np.std(np.array(list_solutions_no_spatial_09[ns])[:,i] - true_params_effective_no_spatial[i]),3))+')' for i in range(len(true_params_effective_no_spatial))] + ["/"]
    columns_1.append(col_to_add)
    col_to_add = [' ['+  str(np.round(mean_squared_error(np.array(list_solutions_no_spatial_09[ns])[:,i], [true_params_effective_no_spatial[i]]*len(list_solutions_no_spatial_09[ns])),3)) +']' for i in range(len(true_params_effective_no_spatial))] + ["/"]
    columns_2.append(col_to_add)
        
columns_1.insert(0,param_names)
columns_2.insert(0,param_names_2)

tr_columns_1 = np.transpose(columns_1)
tr_columns_2 = np.transpose(columns_2)
for i in range(len(tr_columns_1)):
    table_params.add_row(tr_columns_1[i])
    table_params.add_row(tr_columns_2[i])

print(table_params.draw())

+-----------+-----------+----------+----------+----------+----------+----------+
| Parameter | S n=50    | S n=200  | S n=1000 | NS n=50  | NS n=200 | NS       |
|           |           |          |          |          |          | n=1000   |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | 0.007     | 0.011    | 0.01     | 0.096    | 0.019    | 0.044    |
| {2}{*}{$\ | (0.076)   | (0.021)  | (0.011)  | (0.596)  | (0.24)   | (0.097)  |
| beta_{01} |           |          |          |          |          |          |
| $}        |           |          |          |          |          |          |
+-----------+-----------+----------+----------+----------+----------+----------+
|           | [0.006]   | [0.001]  | [0.0]    |  [0.364] |  [0.058] |  [0.011] |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | -0.022    | -0.019   | -0.024   | 0.303    | 0.298    | 0.206    |
| {2}{*}{$\ | (0.126)   | (0

In [275]:
print(latextable.draw_latex(table_params, caption="Difference between the estimated parameters and the true parameters ($rho=0.9$). The results are presented as the mean of the differences on the 100 iterations, the standard deviation within parenthesis, and the standard deviation within square brackets."))

\begin{table}
	\begin{center}
		\begin{tabular}{|l|l|l|l|l|l|l|}
			\hline
			 \\
			\hline
			Parameter & S n=50 & S n=200 & S n=1000 & NS n=50 & NS n=200 & NS n=1000 \\
			\hline
			\multirow{2}{*}{$\beta_{01}$} & 0.007 (0.076) & 0.011 (0.021) & 0.01 (0.011) & 0.096 (0.596) & 0.019 (0.24) & 0.044 (0.097) \\
			\hline
			 & [0.006] & [0.001] & [0.0] &  [0.364] &  [0.058] &  [0.011] \\
			\hline
			\multirow{2}{*}{$\beta_{02}$} & -0.022 (0.126) & -0.019 (0.03) & -0.024 (0.017) & 0.303 (1.096) & 0.298 (0.381) & 0.206 (0.165) \\
			\hline
			 & [0.016] & [0.001] & [0.001] &  [1.292] &  [0.234] &  [0.069] \\
			\hline
			\multirow{2}{*}{$\beta_{11}$} & -0.418 (0.335) & -0.423 (0.194) & -0.39 (0.112) & -0.653 (0.254) & -0.696 (0.113) & -0.673 (0.05) \\
			\hline
			 & [0.287] & [0.217] & [0.165] &  [0.491] &  [0.496] &  [0.455] \\
			\hline
			\multirow{2}{*}{$\beta_{12}$} & 1.193 (0.481) & 1.185 (0.345) & 1.148 (0.218) & 1.588 (0.27) & 1.559 (0.088) & 1.531 (0.049) \\
			\hline
			 & [1.6

In [22]:
table_params = Texttable()
table_params.add_row(["Parameter", "S n=50", "S n=200", "S n=1000",
                      "NS n=50", "NS n=200", "NS n=1000"])

param_names = ["\\multirow{2}{*}{$\\beta_{01}$}", "\\multirow{2}{*}{$\\beta_{02}$}", "\\multirow{2}{*}{$\\beta_{11}$}",
               "\\multirow{2}{*}{$\\beta_{12}$}", "\\multirow{2}{*}{$\\beta_{21}$}", "\\multirow{2}{*}{$\\beta_{22}$}",
               "\\multirow{2}{*}{$\\gamma_0$}", "\\multirow{2}{*}{$\\gamma_1$}", "\\multirow{2}{*}{$\\rho$}" ]
param_names_2 = [""]*len(param_names)

columns_1 = []
columns_2 = []

for ns in range(3):
    col_to_add = [str(np.round(np.mean(list_solutions_spatial_00[ns][:,i] - true_params_effective_00[i]),3))
                  +' ('+str(np.round(np.std(list_solutions_spatial_00[ns][:,i] - true_params_effective_00[i]),3))+')' for i in range(len(true_params_effective_00))]
    columns_1.append(col_to_add)
    col_to_add = ['['+  str(np.round(mean_squared_error(list_solutions_spatial_00[ns][:,i], [true_params_effective_00[i]]*100),3)) +']' for i in range(len(true_params_effective_00))]
    columns_2.append(col_to_add)
    
for ns in range(3):
    col_to_add = [str(np.round(np.mean(list_solutions_no_spatial_00[ns][:,i] - true_params_effective_no_spatial[i]),3))
                  +' ('+str(np.round(np.std(list_solutions_no_spatial_00[ns][:,i] - true_params_effective_no_spatial[i]),3))+')' for i in range(len(true_params_effective_no_spatial))] + ["/"]
    columns_1.append(col_to_add)
    col_to_add = [' ['+  str(np.round(mean_squared_error(list_solutions_no_spatial_00[ns][:,i], [true_params_effective_no_spatial[i]]*100),3)) +']' for i in range(len(true_params_effective_no_spatial))] + ["/"]
    columns_2.append(col_to_add)
        
columns_1.insert(0,param_names)
columns_2.insert(0,param_names_2)

tr_columns_1 = np.transpose(columns_1)
tr_columns_2 = np.transpose(columns_2)
for i in range(len(tr_columns_1)):
    table_params.add_row(tr_columns_1[i])
    table_params.add_row(tr_columns_2[i])

print(table_params.draw())

+-----------+-----------+----------+----------+----------+----------+----------+
| Parameter | S n=50    | S n=200  | S n=1000 | NS n=50  | NS n=200 | NS       |
|           |           |          |          |          |          | n=1000   |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | 0.649     | 0.553    | 0.461    | 0.617    | 0.543    | 0.46     |
| {2}{*}{$\ | (0.418)   | (0.203)  | (0.087)  | (0.393)  | (0.201)  | (0.085)  |
| beta_{01} |           |          |          |          |          |          |
| $}        |           |          |          |          |          |          |
+-----------+-----------+----------+----------+----------+----------+----------+
|           | [0.596]   | [0.347]  | [0.221]  |  [0.535] |  [0.335] |  [0.219] |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | -0.384    | -0.359   | -0.313   | -0.342   | -0.349   | -0.311   |
| {2}{*}{$\ | (0.433)   | (0

### Results on cross-entropy

In [19]:
true_params_effective_01_ce = np.concatenate([beta[:,1:].flatten(), [0.1]])
true_params_effective_05_ce = np.concatenate([beta[:,1:].flatten(), [0.5]])
true_params_effective_09_ce = np.concatenate([beta[:,1:].flatten(), [0.9]])
true_params_effective_no_spatial_ce = np.concatenate([beta[:,1:].flatten()])

In [20]:
table_params = Texttable()
table_params.add_row(["Parameter", "S n=50", "S n=200", "S n=1000",
                      "NS n=50", "NS n=200", "NS n=1000"])
                      
#param_names = ["$\\beta_{01}$", "", "$\\beta_{02}$", "", "$\\beta_{11}$", "", "$\\beta_{12}$", "",
#               "$\\beta_{21}$", "", "$\\beta_{22}$", "", "$\\gamma_0$", "", "$\\gamma_1$", "", "$\\rho$", ""]

param_names = ["\\multirow{2}{*}{$\\beta_{01}$}", "\\multirow{2}{*}{$\\beta_{02}$}", "\\multirow{2}{*}{$\\beta_{11}$}",
               "\\multirow{2}{*}{$\\beta_{12}$}", "\\multirow{2}{*}{$\\beta_{21}$}", "\\multirow{2}{*}{$\\beta_{22}$}",
               "\\multirow{2}{*}{$\\rho$}" ]
param_names_2 = [""]*len(param_names)

columns_1 = []
columns_2 = []

for ns in range(3):
    col_to_add = [str(np.round(np.mean(list_solutions_ce_spatial_01[ns][:,i] - true_params_effective_01_ce[i]),3))
                  +' ('+str(np.round(np.std(list_solutions_ce_spatial_01[ns][:,i] - true_params_effective_01_ce[i]),3))+')' for i in range(len(true_params_effective_01_ce))]
    columns_1.append(col_to_add)
    col_to_add = ['['+  str(np.round(mean_squared_error(list_solutions_ce_spatial_01[ns][:,i], [true_params_effective_01_ce[i]]*100),3)) +']' for i in range(len(true_params_effective_01_ce))]
    columns_2.append(col_to_add)
    
for ns in range(3):
    col_to_add = [str(np.round(np.mean(list_solutions_ce_no_spatial_01[ns][:,i] - true_params_effective_no_spatial_ce[i]),3))
                  +' ('+str(np.round(np.std(list_solutions_ce_no_spatial_01[ns][:,i] - true_params_effective_no_spatial_ce[i]),3))+')' for i in range(len(true_params_effective_no_spatial_ce))] + ["/"]
    columns_1.append(col_to_add)
    col_to_add = [' ['+  str(np.round(mean_squared_error(list_solutions_ce_no_spatial_01[ns][:,i], [true_params_effective_no_spatial_ce[i]]*100),3)) +']' for i in range(len(true_params_effective_no_spatial_ce))] + ["/"]
    columns_2.append(col_to_add)
        
columns_1.insert(0,param_names)
columns_2.insert(0,param_names_2)

tr_columns_1 = np.transpose(columns_1)
tr_columns_2 = np.transpose(columns_2)
for i in range(len(tr_columns_1)):
    table_params.add_row(tr_columns_1[i])
    table_params.add_row(tr_columns_2[i])

print(table_params.draw())

+-----------+-----------+----------+----------+----------+----------+----------+
| Parameter | S n=50    | S n=200  | S n=1000 | NS n=50  | NS n=200 | NS       |
|           |           |          |          |          |          | n=1000   |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | 0.019     | 0.002    | 0.002    | 0.02     | 0.0      | 0.002    |
| {2}{*}{$\ | (0.088)   | (0.051)  | (0.022)  | (0.102)  | (0.058)  | (0.025)  |
| beta_{01} |           |          |          |          |          |          |
| $}        |           |          |          |          |          |          |
+-----------+-----------+----------+----------+----------+----------+----------+
|           | [0.008]   | [0.003]  | [0.0]    |  [0.011] |  [0.003] |  [0.001] |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | 0.021     | 0.004    | 0.003    | 0.037    | 0.022    | 0.016    |
| {2}{*}{$\ | (0.108)   | (0

In [21]:
print(latextable.draw_latex(table_params, caption="Difference between the estimated parameters with the cross-entropy minimization and the true parameters ($rho=0.1$). The results are presented as the mean of the differences on the 100 iterations, the standard deviation within parenthesis, and the standard deviation within square brackets."))

\begin{table}
	\begin{center}
		\begin{tabular}{|l|l|l|l|l|l|l|}
			\hline
			 \\
			\hline
			Parameter & S n=50 & S n=200 & S n=1000 & NS n=50 & NS n=200 & NS n=1000 \\
			\hline
			\multirow{2}{*}{$\beta_{01}$} & 0.019 (0.088) & 0.002 (0.051) & 0.002 (0.022) & 0.02 (0.102) & 0.0 (0.058) & 0.002 (0.025) \\
			\hline
			 & [0.008] & [0.003] & [0.0] &  [0.011] &  [0.003] &  [0.001] \\
			\hline
			\multirow{2}{*}{$\beta_{02}$} & 0.021 (0.108) & 0.004 (0.048) & 0.003 (0.022) & 0.037 (0.13) & 0.022 (0.056) & 0.016 (0.027) \\
			\hline
			 & [0.012] & [0.002] & [0.0] &  [0.018] &  [0.004] &  [0.001] \\
			\hline
			\multirow{2}{*}{$\beta_{11}$} & -0.052 (0.117) & -0.02 (0.051) & -0.002 (0.023) & -0.052 (0.117) & -0.018 (0.051) & -0.001 (0.024) \\
			\hline
			 & [0.016] & [0.003] & [0.001] &  [0.016] &  [0.003] &  [0.001] \\
			\hline
			\multirow{2}{*}{$\beta_{12}$} & 0.141 (0.127) & 0.043 (0.076) & 0.008 (0.035) & 0.145 (0.129) & 0.046 (0.077) & 0.013 (0.036) \\
			\hline
			 & [0.036] 

In [22]:
table_params = Texttable()
table_params.add_row(["Parameter", "S n=50", "S n=200", "S n=1000",
                      "NS n=50", "NS n=200", "NS n=1000"])
                      
#param_names = ["$\\beta_{01}$", "", "$\\beta_{02}$", "", "$\\beta_{11}$", "", "$\\beta_{12}$", "",
#               "$\\beta_{21}$", "", "$\\beta_{22}$", "", "$\\gamma_0$", "", "$\\gamma_1$", "", "$\\rho$", ""]

param_names = ["\\multirow{2}{*}{$\\beta_{01}$}", "\\multirow{2}{*}{$\\beta_{02}$}", "\\multirow{2}{*}{$\\beta_{11}$}",
               "\\multirow{2}{*}{$\\beta_{12}$}", "\\multirow{2}{*}{$\\beta_{21}$}", "\\multirow{2}{*}{$\\beta_{22}$}",
               "\\multirow{2}{*}{$\\rho$}" ]
param_names_2 = [""]*len(param_names)

columns_1 = []
columns_2 = []

for ns in range(3):
    col_to_add = [str(np.round(np.mean(np.array(list_solutions_ce_spatial_05[ns])[:,i] - true_params_effective_05_ce[i]),3))
                  +' ('+str(np.round(np.std(np.array(list_solutions_ce_spatial_05[ns])[:,i] - true_params_effective_05_ce[i]),3))+')' for i in range(len(true_params_effective_05_ce))]
    columns_1.append(col_to_add)
    col_to_add = ['['+  str(np.round(mean_squared_error(np.array(list_solutions_ce_spatial_05[ns])[:,i], [true_params_effective_05_ce[i]]*len(list_solutions_ce_spatial_05[ns])),3)) +']' for i in range(len(true_params_effective_05_ce))]
    columns_2.append(col_to_add)
    
for ns in range(3):
    col_to_add = [str(np.round(np.mean(np.array(list_solutions_ce_no_spatial_05[ns])[:,i] - true_params_effective_no_spatial_ce[i]),3))
                  +' ('+str(np.round(np.std(np.array(list_solutions_ce_no_spatial_05[ns])[:,i] - true_params_effective_no_spatial_ce[i]),3))+')' for i in range(len(true_params_effective_no_spatial_ce))] + ["/"]
    columns_1.append(col_to_add)
    col_to_add = [' ['+  str(np.round(mean_squared_error(np.array(list_solutions_ce_no_spatial_05[ns])[:,i], [true_params_effective_no_spatial_ce[i]]*len(list_solutions_ce_no_spatial_05[ns])),3)) +']' for i in range(len(true_params_effective_no_spatial_ce))] + ["/"]
    columns_2.append(col_to_add)
        
columns_1.insert(0,param_names)
columns_2.insert(0,param_names_2)

tr_columns_1 = np.transpose(columns_1)
tr_columns_2 = np.transpose(columns_2)
for i in range(len(tr_columns_1)):
    table_params.add_row(tr_columns_1[i])
    table_params.add_row(tr_columns_2[i])

print(table_params.draw())

+-----------+-----------+----------+----------+----------+----------+----------+
| Parameter | S n=50    | S n=200  | S n=1000 | NS n=50  | NS n=200 | NS       |
|           |           |          |          |          |          | n=1000   |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | -0.001    | 0.006    | 0.001    | 0.013    | 0.015    | 0.022    |
| {2}{*}{$\ | (0.058)   | (0.031)  | (0.013)  | (0.216)  | (0.115)  | (0.051)  |
| beta_{01} |           |          |          |          |          |          |
| $}        |           |          |          |          |          |          |
+-----------+-----------+----------+----------+----------+----------+----------+
|           | [0.003]   | [0.001]  | [0.0]    |  [0.047] |  [0.013] |  [0.003] |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | -0.005    | 0.007    | 0.001    | 0.123    | 0.206    | 0.169    |
| {2}{*}{$\ | (0.062)   | (0

In [23]:
print(latextable.draw_latex(table_params, caption="Difference between the estimated parameters with the cross-entropy minimization and the true parameters ($rho=0.5$). The results are presented as the mean of the differences on the 100 iterations, the standard deviation within parenthesis, and the standard deviation within square brackets."))

\begin{table}
	\begin{center}
		\begin{tabular}{|l|l|l|l|l|l|l|}
			\hline
			 \\
			\hline
			Parameter & S n=50 & S n=200 & S n=1000 & NS n=50 & NS n=200 & NS n=1000 \\
			\hline
			\multirow{2}{*}{$\beta_{01}$} & -0.001 (0.058) & 0.006 (0.031) & 0.001 (0.013) & 0.013 (0.216) & 0.015 (0.115) & 0.022 (0.051) \\
			\hline
			 & [0.003] & [0.001] & [0.0] &  [0.047] &  [0.013] &  [0.003] \\
			\hline
			\multirow{2}{*}{$\beta_{02}$} & -0.005 (0.062) & 0.007 (0.027) & 0.001 (0.013) & 0.123 (0.377) & 0.206 (0.195) & 0.169 (0.086) \\
			\hline
			 & [0.004] & [0.001] & [0.0] &  [0.157] &  [0.08] &  [0.036] \\
			\hline
			\multirow{2}{*}{$\beta_{11}$} & -0.036 (0.104) & -0.011 (0.052) & -0.002 (0.026) & -0.026 (0.139) & 0.001 (0.069) & 0.007 (0.033) \\
			\hline
			 & [0.012] & [0.003] & [0.001] &  [0.02] &  [0.005] &  [0.001] \\
			\hline
			\multirow{2}{*}{$\beta_{12}$} & 0.194 (0.121) & 0.055 (0.082) & 0.015 (0.036) & 0.314 (0.211) & 0.211 (0.112) & 0.204 (0.054) \\
			\hline
			 & [0.05

In [24]:
table_params = Texttable()
table_params.add_row(["Parameter", "S n=50", "S n=200", "S n=1000",
                      "NS n=50", "NS n=200", "NS n=1000"])
                      
#param_names = ["$\\beta_{01}$", "", "$\\beta_{02}$", "", "$\\beta_{11}$", "", "$\\beta_{12}$", "",
#               "$\\beta_{21}$", "", "$\\beta_{22}$", "", "$\\gamma_0$", "", "$\\gamma_1$", "", "$\\rho$", ""]

param_names = ["\\multirow{2}{*}{$\\beta_{01}$}", "\\multirow{2}{*}{$\\beta_{02}$}", "\\multirow{2}{*}{$\\beta_{11}$}",
               "\\multirow{2}{*}{$\\beta_{12}$}", "\\multirow{2}{*}{$\\beta_{21}$}", "\\multirow{2}{*}{$\\beta_{22}$}",
               "\\multirow{2}{*}{$\\rho$}" ]
param_names_2 = [""]*len(param_names)

columns_1 = []
columns_2 = []

for ns in range(3):
    col_to_add = [str(np.round(np.mean(np.array(list_solutions_ce_spatial_09[ns])[:,i] - true_params_effective_09_ce[i]),3))
                  +' ('+str(np.round(np.std(np.array(list_solutions_ce_spatial_09[ns])[:,i] - true_params_effective_09_ce[i]),3))+')' for i in range(len(true_params_effective_09_ce))]
    columns_1.append(col_to_add)
    col_to_add = ['['+  str(np.round(mean_squared_error(np.array(list_solutions_ce_spatial_09[ns])[:,i], [true_params_effective_09_ce[i]]*len(list_solutions_ce_spatial_09[ns])),3)) +']' for i in range(len(true_params_effective_09_ce))]
    columns_2.append(col_to_add)
    
for ns in range(3):
    col_to_add = [str(np.round(np.mean(np.array(list_solutions_ce_no_spatial_09[ns])[:,i] - true_params_effective_no_spatial_ce[i]),3))
                  +' ('+str(np.round(np.std(np.array(list_solutions_ce_no_spatial_09[ns])[:,i] - true_params_effective_no_spatial_ce[i]),3))+')' for i in range(len(true_params_effective_no_spatial_ce))] + ["/"]
    columns_1.append(col_to_add)
    col_to_add = [' ['+  str(np.round(mean_squared_error(np.array(list_solutions_ce_no_spatial_09[ns])[:,i], [true_params_effective_no_spatial_ce[i]]*len(list_solutions_ce_no_spatial_09[ns])),3)) +']' for i in range(len(true_params_effective_no_spatial_ce))] + ["/"]
    columns_2.append(col_to_add)
        
columns_1.insert(0,param_names)
columns_2.insert(0,param_names_2)

tr_columns_1 = np.transpose(columns_1)
tr_columns_2 = np.transpose(columns_2)
for i in range(len(tr_columns_1)):
    table_params.add_row(tr_columns_1[i])
    table_params.add_row(tr_columns_2[i])

print(table_params.draw())

+-----------+-----------+----------+----------+----------+----------+----------+
| Parameter | S n=50    | S n=200  | S n=1000 | NS n=50  | NS n=200 | NS       |
|           |           |          |          |          |          | n=1000   |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | 0.002     | 0.0      | 0.0      | 0.143    | 0.062    | 0.179    |
| {2}{*}{$\ | (0.063)   | (0.009)  | (0.004)  | (1.442)  | (0.73)   | (0.292)  |
| beta_{01} |           |          |          |          |          |          |
| $}        |           |          |          |          |          |          |
+-----------+-----------+----------+----------+----------+----------+----------+
|           | [0.004]   | [0.0]    | [0.0]    |  [2.099] |  [0.537] |  [0.117] |
+-----------+-----------+----------+----------+----------+----------+----------+
| \multirow | -0.004    | -0.003   | -0.0     | 0.696    | 0.845    | 0.726    |
| {2}{*}{$\ | (0.088)   | (0

In [25]:
print(latextable.draw_latex(table_params, caption="Difference between the estimated parameters with the cross-entropy minimization and the true parameters ($rho=0.9$). The results are presented as the mean of the differences on the 100 iterations, the standard deviation within parenthesis, and the standard deviation within square brackets."))

\begin{table}
	\begin{center}
		\begin{tabular}{|l|l|l|l|l|l|l|}
			\hline
			 \\
			\hline
			Parameter & S n=50 & S n=200 & S n=1000 & NS n=50 & NS n=200 & NS n=1000 \\
			\hline
			\multirow{2}{*}{$\beta_{01}$} & 0.002 (0.063) & 0.0 (0.009) & 0.0 (0.004) & 0.143 (1.442) & 0.062 (0.73) & 0.179 (0.292) \\
			\hline
			 & [0.004] & [0.0] & [0.0] &  [2.099] &  [0.537] &  [0.117] \\
			\hline
			\multirow{2}{*}{$\beta_{02}$} & -0.004 (0.088) & -0.003 (0.01) & -0.0 (0.004) & 0.696 (1.836) & 0.845 (0.743) & 0.726 (0.34) \\
			\hline
			 & [0.008] & [0.0] & [0.0] &  [3.853] &  [1.267] &  [0.643] \\
			\hline
			\multirow{2}{*}{$\beta_{11}$} & -0.188 (0.316) & -0.029 (0.054) & -0.003 (0.023) & -0.22 (0.37) & -0.179 (0.155) & -0.218 (0.067) \\
			\hline
			 & [0.135] & [0.004] & [0.001] &  [0.185] &  [0.056] &  [0.052] \\
			\hline
			\multirow{2}{*}{$\beta_{12}$} & 0.643 (0.355) & 0.164 (0.112) & 0.033 (0.038) & 1.109 (0.426) & 1.291 (0.199) & 1.32 (0.099) \\
			\hline
			 & [0.54] & [0.039]

# Analysis of the results on a test set

In [16]:
def analysis_results(list_solutions_spatial, list_solutions_no_spatial, rho,
                     n_repeat=100, size_test=1000, list_n_samples=[50,200,1000]):
    
    list_r2_s, list_rmse_s, list_similarity_s, list_crossentropy_s, list_rmse_a_s = [], [], [], [], []
    list_r2_ns, list_rmse_ns, list_similarity_ns, list_crossentropy_ns, list_rmse_a_ns = [], [], [], [], []

    seed=0

    for n_samples_index in range(3):
        n_samples = list_n_samples[n_samples_index]
        
        list_r2_s_temp, list_rmse_s_temp, list_similarity_s_temp, list_crossentropy_s_temp, list_rmse_a_s_temp = [], [], [], [], []
        list_r2_ns_temp, list_rmse_ns_temp, list_similarity_ns_temp, list_crossentropy_ns_temp, list_rmse_a_ns_temp = [], [], [], [], []
        
        for i in range(len(list_solutions_spatial[n_samples_index])):
            np.random.seed(seed+1000)

            X,Z,W = create_features_matrices(n_samples,n_features,choice_W='random_distance',nneighbors=10)
            Z[:,0] = 1
            M = np.identity(n_samples) - rho*W
            try:
                mu = dirichlet_regression.compute_mu_spatial(X, beta, M)
                phi = np.exp(np.matmul(Z,gamma_var))
                alpha = mu*phi[:,None]

                Y = np.array([np.random.dirichlet(alpha_i) for alpha_i in alpha])
                Y = (Y*(size_test-1)+1/n_classes)/size_test

                solution_spatial = list_solutions_spatial[n_samples_index][i]
                beta_sol_s = np.zeros((n_features+1,n_classes))
                beta_sol_s[:,1:] = solution_spatial[:(n_features+1)*(n_classes-1)].reshape((n_features+1),n_classes-1)
                rho_sol_s = solution_spatial[-1]
                M_sol = np.identity(n_samples) - rho_sol_s*W
                mu_sol_s = dirichlet_regression.compute_mu_spatial(X, beta_sol_s, M_sol)

                list_r2_s_temp.append(r2_score(Y,mu_sol_s))
                list_rmse_s_temp.append(mean_squared_error(Y,mu_sol_s,squared=False))
                list_crossentropy_s_temp.append(-(1/size_test)*np.sum(Y*np.log(mu_sol_s)))
                list_similarity_s_temp.append(cos_similarity(Y, mu_sol_s))
                list_rmse_a_s_temp.append(rmse_aitchison(Y,mu_sol_s))

                solution_no_spatial = list_solutions_no_spatial[n_samples_index][i]
                beta_sol_ns = np.zeros((n_features+1,n_classes))
                beta_sol_ns[:,1:] = solution_no_spatial[:(n_features+1)*(n_classes-1)].reshape((n_features+1),n_classes-1)
                mu_sol_ns = dirichlet_regression.compute_mu(X, beta_sol_ns)

                list_r2_ns_temp.append(r2_score(Y,mu_sol_ns))
                list_rmse_ns_temp.append(mean_squared_error(Y,mu_sol_ns,squared=False))
                list_crossentropy_ns_temp.append(-(1/size_test)*np.sum(Y*np.log(mu_sol_ns)))
                list_similarity_ns_temp.append(cos_similarity(Y, mu_sol_ns))
                list_rmse_a_ns_temp.append(rmse_aitchison(Y,mu_sol_ns))

            except RuntimeError:
                print("Factor is exactly singular")
            except np.linalg.LinAlgError:
                print("Singular matrix")

            seed+=1
            
        list_r2_s.append(list_r2_s_temp)
        list_rmse_s.append(list_rmse_s_temp)
        list_similarity_s.append(list_similarity_s_temp)
        list_crossentropy_s.append(list_crossentropy_s_temp)
        list_rmse_a_s.append(list_rmse_a_s_temp)
        
        list_r2_ns.append(list_r2_ns_temp)
        list_rmse_ns.append(list_rmse_ns_temp)
        list_similarity_ns.append(list_similarity_ns_temp)
        list_crossentropy_ns.append(list_crossentropy_ns_temp)
        list_rmse_a_ns.append(list_rmse_a_ns_temp)
        
    return(list_r2_s, list_rmse_s, list_similarity_s, list_crossentropy_s, list_rmse_a_s,
           list_r2_ns, list_rmse_ns, list_similarity_ns, list_crossentropy_ns, list_rmse_a_ns)

## rho=0.1

In [301]:
%%time
list_r2_s, list_rmse_s, list_similarity_s, list_crossentropy_s, list_r2_ns, list_rmse_ns, list_similarity_ns, list_crossentropy_ns = analysis_results(list_solutions_spatial_01, list_solutions_no_spatial_01, rho=0.1) 

Wall time: 10.6 s


In [302]:
print("DIRICHLET no spatial")
print('R2', np.round(np.mean(list_r2_ns[2]),4), '(', np.round(np.var(list_r2_ns[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_ns[2]),4), '(', np.round(np.var(list_rmse_ns[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_ns[2]),4), '(', np.round(np.var(list_crossentropy_ns[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_ns[2]),4), '(', np.round(np.var(list_similarity_ns[2]),4), ')')
print("---\nDIRICHLET spatial")
print('R2', np.round(np.mean(list_r2_s[2]),4), '(', np.round(np.var(list_r2_s[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_s[2]),4), '(', np.round(np.var(list_rmse_s[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_s[2]),4), '(', np.round(np.var(list_crossentropy_s[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_s[2]),4), '(', np.round(np.var(list_similarity_s[2]),4), ')')

DIRICHLET no spatial
R2 0.9309 ( 0.0 )
RMSE 0.0739 ( 0.0 )
Crossentropy 0.666 ( 0.0001 )
Cos similarity 0.9855 ( 0.0 )
---
DIRICHLET spatial
R2 0.9335 ( 0.0 )
RMSE 0.0723 ( 0.0 )
Crossentropy 0.6648 ( 0.0001 )
Cos similarity 0.9862 ( 0.0 )


In [17]:
%%time
list_r2_s, list_rmse_s, list_similarity_s, list_crossentropy_s, list_rmse_a_s, list_r2_ns, list_rmse_ns, list_similarity_ns, list_crossentropy_ns, list_rmse_a_ns = analysis_results(list_solutions_spatial_01, list_solutions_no_spatial_01, rho=0.1) 

Wall time: 11.3 s


In [19]:
print("DIRICHLET no spatial")
print('R2', np.round(np.mean(list_r2_ns[2]),4), '(', np.round(np.var(list_r2_ns[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_ns[2]),4), '(', np.round(np.var(list_rmse_ns[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_ns[2]),4), '(', np.round(np.var(list_crossentropy_ns[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_ns[2]),4), '(', np.round(np.var(list_similarity_ns[2]),4), ')')
print('RMSE_A', np.round(np.mean(list_rmse_a_ns[2]),4), '(', np.round(np.var(list_rmse_a_ns[2]),4), ')')
print("---\nDIRICHLET spatial")
print('R2', np.round(np.mean(list_r2_s[2]),4), '(', np.round(np.var(list_r2_s[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_s[2]),4), '(', np.round(np.var(list_rmse_s[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_s[2]),4), '(', np.round(np.var(list_crossentropy_s[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_s[2]),4), '(', np.round(np.var(list_similarity_s[2]),4), ')')
print('RMSE_A', np.round(np.mean(list_rmse_a_s[2]),4), '(', np.round(np.var(list_rmse_a_s[2]),4), ')')

DIRICHLET no spatial
R2 0.9309 ( 0.0 )
RMSE 0.0739 ( 0.0 )
Crossentropy 0.666 ( 0.0001 )
Cos similarity 0.9855 ( 0.0 )
RMSE_A 1.0007 ( 0.0012 )
---
DIRICHLET spatial
R2 0.9335 ( 0.0 )
RMSE 0.0723 ( 0.0 )
Crossentropy 0.6648 ( 0.0001 )
Cos similarity 0.9862 ( 0.0 )
RMSE_A 0.984 ( 0.0012 )


In [20]:
%%time
list_r2_s, list_rmse_s, list_similarity_s, list_crossentropy_s, list_rmse_a_s, list_r2_ns, list_rmse_ns, list_similarity_ns, list_crossentropy_ns, list_rmse_a_ns = analysis_results(list_solutions_ce_spatial_01, list_solutions_ce_no_spatial_01, rho=0.1)

Wall time: 9.55 s


In [21]:
print("CROSSENTROPY no spatial")
print('R2', np.round(np.mean(list_r2_ns[2]),4), '(', np.round(np.var(list_r2_ns[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_ns[2]),4), '(', np.round(np.var(list_rmse_ns[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_ns[2]),4), '(', np.round(np.var(list_crossentropy_ns[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_ns[2]),4), '(', np.round(np.var(list_similarity_ns[2]),4), ')')
print('RMSE_A', np.round(np.mean(list_rmse_a_ns[2]),4), '(', np.round(np.var(list_rmse_a_ns[2]),4), ')')
print("---\nCROSSENTROPY spatial")
print('R2', np.round(np.mean(list_r2_s[2]),4), '(', np.round(np.var(list_r2_s[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_s[2]),4), '(', np.round(np.var(list_rmse_s[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_s[2]),4), '(', np.round(np.var(list_crossentropy_s[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_s[2]),4), '(', np.round(np.var(list_similarity_s[2]),4), ')')
print('RMSE_A', np.round(np.mean(list_rmse_a_s[2]),4), '(', np.round(np.var(list_rmse_a_s[2]),4), ')')

CROSSENTROPY no spatial
R2 0.9314 ( 0.0 )
RMSE 0.0736 ( 0.0 )
Crossentropy 0.6655 ( 0.0001 )
Cos similarity 0.9856 ( 0.0 )
RMSE_A 0.9571 ( 0.0011 )
---
CROSSENTROPY spatial
R2 0.9338 ( 0.0 )
RMSE 0.072 ( 0.0 )
Crossentropy 0.6644 ( 0.0001 )
Cos similarity 0.9862 ( 0.0 )
RMSE_A 0.9454 ( 0.001 )


## rho=0.5

In [22]:
%%time
list_r2_s, list_rmse_s, list_similarity_s, list_crossentropy_s,  list_rmse_a_s, list_r2_ns, list_rmse_ns, list_similarity_ns, list_crossentropy_ns,  list_rmse_a_ns = analysis_results(list_solutions_spatial_05, list_solutions_no_spatial_05, rho=0.5) 

Wall time: 10.1 s


In [23]:
print("DIRICHLET no spatial")
print('R2', np.round(np.mean(list_r2_ns[2]),4), '(', np.round(np.var(list_r2_ns[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_ns[2]),4), '(', np.round(np.var(list_rmse_ns[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_ns[2]),4), '(', np.round(np.var(list_crossentropy_ns[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_ns[2]),4), '(', np.round(np.var(list_similarity_ns[2]),4), ')')
print('RMSE_A', np.round(np.mean(list_rmse_a_ns[2]),4), '(', np.round(np.var(list_rmse_a_ns[2]),4), ')')
print("---\nDIRICHLET spatial")
print('R2', np.round(np.mean(list_r2_s[2]),4), '(', np.round(np.var(list_r2_s[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_s[2]),4), '(', np.round(np.var(list_rmse_s[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_s[2]),4), '(', np.round(np.var(list_crossentropy_s[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_s[2]),4), '(', np.round(np.var(list_similarity_s[2]),4), ')')
print('RMSE_A', np.round(np.mean(list_rmse_a_s[2]),4), '(', np.round(np.var(list_rmse_a_s[2]),4), ')')

DIRICHLET no spatial
R2 0.8311 ( 0.0001 )
RMSE 0.1249 ( 0.0 )
Crossentropy 0.6845 ( 0.0001 )
Cos similarity 0.961 ( 0.0 )
RMSE_A 1.5856 ( 0.0031 )
---
DIRICHLET spatial
R2 0.9408 ( 0.0 )
RMSE 0.0705 ( 0.0 )
Crossentropy 0.6275 ( 0.0002 )
Cos similarity 0.9872 ( 0.0 )
RMSE_A 1.0458 ( 0.0012 )


In [24]:
list_r2_s, list_rmse_s, list_similarity_s, list_crossentropy_s, list_rmse_a_s, list_r2_ns, list_rmse_ns, list_similarity_ns, list_crossentropy_ns, list_rmse_a_ns = analysis_results(list_solutions_ce_spatial_05, list_solutions_ce_no_spatial_05, rho=0.5)

In [25]:
print("CROSSENTROPY no spatial")
print('R2', np.round(np.mean(list_r2_ns[2]),4), '(', np.round(np.var(list_r2_ns[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_ns[2]),4), '(', np.round(np.var(list_rmse_ns[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_ns[2]),4), '(', np.round(np.var(list_crossentropy_ns[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_ns[2]),4), '(', np.round(np.var(list_similarity_ns[2]),4), ')')
print('RMSE_A', np.round(np.mean(list_rmse_a_ns[2]),4), '(', np.round(np.var(list_rmse_a_ns[2]),4), ')')
print("---\nCROSSENTROPY spatial")
print('R2', np.round(np.mean(list_r2_s[2]),4), '(', np.round(np.var(list_r2_s[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_s[2]),4), '(', np.round(np.var(list_rmse_s[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_s[2]),4), '(', np.round(np.var(list_crossentropy_s[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_s[2]),4), '(', np.round(np.var(list_similarity_s[2]),4), ')')
print('RMSE_A', np.round(np.mean(list_rmse_a_s[2]),4), '(', np.round(np.var(list_rmse_a_s[2]),4), ')')

CROSSENTROPY no spatial
R2 0.8421 ( 0.0001 )
RMSE 0.1207 ( 0.0 )
Crossentropy 0.6764 ( 0.0002 )
Cos similarity 0.9618 ( 0.0 )
RMSE_A 1.3772 ( 0.0025 )
---
CROSSENTROPY spatial
R2 0.9414 ( 0.0 )
RMSE 0.07 ( 0.0 )
Crossentropy 0.627 ( 0.0002 )
Cos similarity 0.9872 ( 0.0 )
RMSE_A 0.9947 ( 0.0011 )


## rho=0.9

In [26]:
%%time
list_r2_s, list_rmse_s, list_similarity_s, list_crossentropy_s, list_rmse_a_s, list_r2_ns, list_rmse_ns, list_similarity_ns, list_crossentropy_ns, list_rmse_a_ns = analysis_results(list_solutions_spatial_09, list_solutions_no_spatial_09, rho=0.9) 

Wall time: 9.82 s


In [27]:
print("DIRICHLET no spatial")
print('R2', np.round(np.mean(list_r2_ns[2]),4), '(', np.round(np.var(list_r2_ns[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_ns[2]),4), '(', np.round(np.var(list_rmse_ns[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_ns[2]),4), '(', np.round(np.var(list_crossentropy_ns[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_ns[2]),4), '(', np.round(np.var(list_similarity_ns[2]),4), ')')
print('RMSE_A', np.round(np.mean(list_rmse_a_ns[2]),4), '(', np.round(np.var(list_rmse_a_ns[2]),4), ')')
print("---\nDIRICHLET spatial")
print('R2', np.round(np.mean(list_r2_s[2]),4), '(', np.round(np.var(list_r2_s[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_s[2]),4), '(', np.round(np.var(list_rmse_s[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_s[2]),4), '(', np.round(np.var(list_crossentropy_s[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_s[2]),4), '(', np.round(np.var(list_similarity_s[2]),4), ')')
print('RMSE_A', np.round(np.mean(list_rmse_a_s[2]),4), '(', np.round(np.var(list_rmse_a_s[2]),4), ')')

DIRICHLET no spatial
R2 0.2073 ( 0.0025 )
RMSE 0.3257 ( 0.0001 )
Crossentropy 0.9033 ( 0.0005 )
Cos similarity 0.7764 ( 0.0001 )
RMSE_A 4.2761 ( 0.0204 )
---
DIRICHLET spatial
R2 0.9011 ( 0.0026 )
RMSE 0.1097 ( 0.0008 )
Crossentropy 0.4414 ( 0.0026 )
Cos similarity 0.9776 ( 0.0001 )
RMSE_A 2.4516 ( 0.1342 )


In [28]:
list_r2_s, list_rmse_s, list_similarity_s, list_crossentropy_s, list_rmse_a_s, list_r2_ns, list_rmse_ns, list_similarity_ns, list_crossentropy_ns, list_rmse_a_ns = analysis_results(list_solutions_ce_spatial_09, list_solutions_ce_no_spatial_09, rho=0.9) 

In [29]:
print("CROSSENTROPY no spatial")
print('R2', np.round(np.mean(list_r2_ns[2]),4), '(', np.round(np.var(list_r2_ns[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_ns[2]),4), '(', np.round(np.var(list_rmse_ns[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_ns[2]),4), '(', np.round(np.var(list_crossentropy_ns[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_ns[2]),4), '(', np.round(np.var(list_similarity_ns[2]),4), ')')
print('RMSE_A', np.round(np.mean(list_rmse_a_ns[2]),4), '(', np.round(np.var(list_rmse_a_ns[2]),4), ')')
print("---\nCROSSENTROPY spatial")
print('R2', np.round(np.mean(list_r2_s[2]),4), '(', np.round(np.var(list_r2_s[2]),4), ')')
print('RMSE', np.round(np.mean(list_rmse_s[2]),4), '(', np.round(np.var(list_rmse_s[2]),4), ')')
print('Crossentropy', np.round(np.mean(list_crossentropy_s[2]),4), '(', np.round(np.var(list_crossentropy_s[2]),4), ')')
print('Cos similarity', np.round(np.mean(list_similarity_s[2]),4), '(', np.round(np.var(list_similarity_s[2]),4), ')')
print('RMSE_A', np.round(np.mean(list_rmse_a_s[2]),4), '(', np.round(np.var(list_rmse_a_s[2]),4), ')')

CROSSENTROPY no spatial
R2 0.274 ( 0.0021 )
RMSE 0.3121 ( 0.0002 )
Crossentropy 0.8576 ( 0.0019 )
Cos similarity 0.7877 ( 0.0003 )
RMSE_A 3.9937 ( 0.0211 )
---
CROSSENTROPY spatial
R2 0.9761 ( 0.0 )
RMSE 0.0535 ( 0.0 )
Crossentropy 0.3716 ( 0.0009 )
Cos similarity 0.9933 ( 0.0 )
RMSE_A 1.9164 ( 0.032 )
